In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import random
import joblib
import torch
import re
import numpy as np
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from datasets import Dataset as HuggingFaceDataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [3]:
file_path = '/content/drive/MyDrive/Resume.csv'
df = pd.read_csv(file_path)

In [4]:
categories_to_remove = [
    'AGRICULTURE', 'APPAREL', 'ARTS', 'CHEF',
    'AVIATION', 'FITNESS', 'PUBLIC-RELATIONS', 'FINANCE'
]

In [5]:
df = df[~df['Category'].isin(categories_to_remove)]
df = df.reset_index(drop=True)
df.drop("ID", axis=1, inplace=True)

print("\n Active Categories:")
print(df['Category'].value_counts())


 Active Categories:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
ACCOUNTANT                118
ENGINEERING               118
SALES                     116
CONSULTANT                115
HEALTHCARE                115
BANKING                   115
CONSTRUCTION              112
HR                        110
DESIGNER                  107
TEACHER                   102
DIGITAL-MEDIA              96
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


In [6]:
target_count = 250
augmented_rows = []

for category in df['Category'].unique():
    cat_df = df[df['Category'] == category]
    current_count = len(cat_df)
    needed = target_count - current_count

    if needed > 0:
        existing_resumes = cat_df['Resume_str'].tolist()
        for i in range(needed):

            base_text = str(random.choice(existing_resumes))
            words = base_text.split()


            new_words = [w for w in words if random.random() > 0.10]

            augmented_rows.append({
                "Resume_str": " ".join(new_words),
                "Resume_html": "",
                "Category": category
            })

if augmented_rows:
    df = pd.concat([df, pd.DataFrame(augmented_rows)], ignore_index=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print("\n New Balanced Categories:")
print(df['Category'].value_counts())


 New Balanced Categories:
Category
ADVOCATE                  250
ENGINEERING               250
BANKING                   250
CONSULTANT                250
HR                        250
DESIGNER                  250
BPO                       250
AUTOMOBILE                250
CONSTRUCTION              250
BUSINESS-DEVELOPMENT      250
INFORMATION-TECHNOLOGY    250
HEALTHCARE                250
DIGITAL-MEDIA             250
TEACHER                   250
SALES                     250
ACCOUNTANT                250
Name: count, dtype: int64


In [7]:
print("\n[INFO] Cleaning Text Data...")
def clean_text(text):
    text = re.sub(r'http\S+\s*', ' ', text)
    text = re.sub(r'RT|cc', ' ', text)
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'@\S+', '  ', text)
    text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@\[\\\]^_`{|}~]', ' ', text)
    text = re.sub(r'[^\x00-\x7f]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

df['Resume_str'] = df['Resume_str'].apply(clean_text)


[INFO] Cleaning Text Data...


In [8]:
print("\n[INFO] Encoding Labels and Splitting Data...")
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])
joblib.dump(le, 'label_encoder.pkl') # Save this file for Streamlit!

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Resume_str'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42
)


[INFO] Encoding Labels and Splitting Data...


In [9]:
print("\n[INFO] Tokenizing Text...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)

train_dataset = HuggingFaceDataset.from_dict({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask'],
    'labels': train_labels
})

val_dataset = HuggingFaceDataset.from_dict({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask'],
    'labels': val_labels
})

train_dataset.set_format('torch')
val_dataset.set_format('torch')


[INFO] Tokenizing Text...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [10]:
print("\n[INFO] Setting up Model & Training Arguments...")
num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=num_labels)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=6,
    learning_rate=2e-5,
    warmup_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

print("\n🚀 Training started! Grab a coffee, this will take a few minutes...\n")
trainer.train()


[INFO] Setting up Model & Training Arguments...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2400, training_loss=0.40183626384474336, metrics={'train_runtime': 1115.1275, 'train_samples_per_second': 17.218, 'train_steps_per_second': 2.152, 'total_flos': 2544009058713600.0, 'train_loss': 0.40183626384474336, 'epoch': 6.0})

In [11]:
print("\n[INFO] Saving the trained model...")
model.save_pretrained('./distilbert_resume_model')
tokenizer.save_pretrained('./distilbert_resume_model')

print("\n[INFO] Calculating Final Accuracy...")
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
acc = accuracy_score(val_labels, preds)

print(f"\n🏆 NEW TOTAL ACCURACY: {acc * 100:.2f}%\n")
print(classification_report(val_labels, preds, target_names=le.classes_))
print("\n✅ All Done! Download your 'label_encoder.pkl' and zip the 'distilbert_resume_model' folder.")


[INFO] Saving the trained model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


[INFO] Calculating Final Accuracy...



🏆 NEW TOTAL ACCURACY: 96.62%

                        precision    recall  f1-score   support

            ACCOUNTANT       0.98      1.00      0.99        54
              ADVOCATE       0.92      0.88      0.90        56
            AUTOMOBILE       0.90      1.00      0.95        61
               BANKING       0.98      0.92      0.95        50
                   BPO       0.95      0.98      0.97        58
  BUSINESS-DEVELOPMENT       0.98      1.00      0.99        50
          CONSTRUCTION       1.00      0.92      0.96        38
            CONSULTANT       0.95      0.97      0.96        40
              DESIGNER       1.00      1.00      1.00        49
         DIGITAL-MEDIA       0.98      0.98      0.98        44
           ENGINEERING       1.00      0.96      0.98        52
            HEALTHCARE       0.89      0.92      0.90        51
                    HR       1.00      1.00      1.00        50
INFORMATION-TECHNOLOGY       0.98      0.92      0.95        49
        